<a href="https://colab.research.google.com/github/Raksh1707/DeepLearning/blob/main/DLDLdl3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install datasets

from datasets import load_dataset
import pandas as pd

raw_dataset=load_dataset("wangrongsheng/ag_news")

df=pd.DataFrame(raw_dataset["train"])
print(df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [27]:
from datasets import load_dataset
import re
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


In [28]:
raw_dataset = load_dataset("wangrongsheng/ag_news")

train_data = raw_dataset["train"]
test_data = raw_dataset["test"]

print("Training samples:", len(train_data))     #display the sample
print("Testing samples:", len(test_data))

Training samples: 120000
Testing samples: 7600


In [29]:
print("\nFirst Original Text:")
print(train_data["text"][0])

print("\nFirst Cleaned Text:")  #cleanind train and test data
print(X_train[0])

print("\nFirst 5 Labels:")
print(y_train[:5])


First Original Text:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

First Cleaned Text:
wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again

First 5 Labels:
[2, 2, 2, 2, 2]


In [30]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")#out of vocabulary
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train) #con
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [31]:
max_length = 50

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

In [32]:

num_classes = 4

y_train_cat = to_categorical(
    y_train,
    num_classes=num_classes
)

y_test_cat = to_categorical(
    y_test,
    num_classes=num_classes
)

print("\n"+"="*50)
print("STEP 5 : ONE-HOT ENCODING")
print("="*50)

print("\nOriginal Labels")
print(y_train[:5])

print("\nEncoded Labels")
print(y_train_cat[:5])


STEP 5 : ONE-HOT ENCODING

Original Labels
[2, 2, 2, 2, 2]

Encoded Labels
[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [33]:
model = Sequential([
    Embedding(input_dim=vocab_size,
              output_dim=64,
              input_length=max_length),

    SimpleRNN(64),

    Dense(num_classes, activation='softmax')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [34]:
model.compile(
    optimizer='adam',                    # predict  weight of model
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_pad,
    y_train_cat,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)


Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 53s 32ms/step - accuracy: 0.7895 - loss: 0.5895 - val_accuracy: 0.8273 - val_loss: 0.4926
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 43s 28ms/step - accuracy: 0.8648 - loss: 0.4077 - val_accuracy: 0.8423 - val_loss: 0.4631
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 47s 32ms/step - accuracy: 0.8888 - loss: 0.3476 - val_accuracy: 0.8505 - val_loss: 0.4477
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 48s 32ms/step - accuracy: 0.9038 - loss: 0.3058 - val_accuracy: 0.8798 - val_loss: 0.3710
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 79s 30ms/step - accuracy: 0.8984 - loss: 0.3294 - val_accuracy: 0.8670 - val_loss: 0.4003


In [36]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (64, 50, 64)           │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (64, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (64, 4)                │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,945,550 (7.42 MB)

 Trainable params: 648,516 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,297,034 (4.95 MB)

In [35]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test_cat
)

print("\n"+"="*50)
print("STEP 8 : MODEL EVALUATION")
print("="*50)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8796 - loss: 0.3705

Test Loss: 0.3705175817012787
Test Accuracy: 0.879605233669281
